<a href="https://colab.research.google.com/github/Liping-LZ/BDAO_DSDO/blob/main/Week%201/05_web_scraping_to_AI_Insights_using_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Web Scraping to AI Insights using API

### What we'll do in this demo:

| Step | What happens | Why it matters |
|------|-------------|----------------|
| **1. Web Scraping** | Collect product data from an eCommerce website automatically | This is how businesses collect data at scale — no copy-pasting |
| **2. Data Preview** | See what the raw data looks like | Raw data could be messy — this is reality |
| **3. Gemini API** | Send the data to Google's AI and ask business questions | This is the simplied pipeline powered by AI in action: collect → analyse → insight |

> **The big idea:** A task that would take a human hours to do manually — visiting hundreds of product pages, taking notes, spotting patterns — takes a computer seconds. That's Big Data.

---
## Step 0: Install & Import Libraries

Libraries are like tools in a toolbox. We need:
- `requests` — to visit websites and fetch their content
- `BeautifulSoup` — to read and extract information from a webpage
- `google-generativeai` — to talk to Google's Gemini AI
- `pandas` — to organise data into a table

In [ ]:
# Install required libraries (run this once)
!pip install requests beautifulsoup4 google-genai pandas -q

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google import genai
from IPython.display import display, Markdown

print("✅ All libraries loaded successfully!")

---
## Step 1: Web Scraping — Collecting Product Data

We're going to visit an online bookshop and automatically collect:
- Book titles
- Prices
- Ratings
- Categories

**The website:** [books.toscrape.com](http://books.toscrape.com) — a demo eCommerce site built specifically for learning data collection.

> **Think about it:** Amazon has over 350 million products. Imagine trying to collect all that data manually. Web scraping makes this possible.

In [ ]:
# --- WEB SCRAPING ---
# We'll collect books from multiple categories to simulate an eCommerce catalogue

BASE_URL = "http://books.toscrape.com/catalogue/"
SITE_URL = "http://books.toscrape.com/"

# Rating words to numbers (the website stores ratings as words!)
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def scrape_books(num_pages):
    """Visit the bookshop and collect product data from multiple pages."""
    all_books = []

    print(f"Starting to collect data from {SITE_URL}")
    print(f"Scanning {num_pages} pages of products...\n")

    for page in range(1, num_pages + 1):
        # Build the URL for each page
        url = f"{SITE_URL}catalogue/page-{page}.html"

        # Visit the webpage (like opening a browser)
        response = requests.get(url)

        # Parse the page content
        soup = BeautifulSoup(response.content, "html.parser")

        # Find all product cards on the page
        books = soup.find_all("article", class_="product_pod")

        for book in books:
            # Extract each piece of information
            title = book.h3.a["title"]
            price = book.find("p", class_="price_color").text.strip()
            rating_word = book.p["class"][1]
            rating = RATING_MAP.get(rating_word, 0)

            all_books.append({
                "Title": title,
                "Price (£)": float(price.replace("£", "").replace("Â", "")),
                "Rating (out of 5)": rating,
            })

        print(f"   ✅ Page {page} — collected {len(books)} products")

    print(f"\n Done! Collected {len(all_books)} products in total.")
    return all_books

# Run the scraper
books_data = scrape_books(num_pages=50)

---
## Step 2: Preview the Raw Data

Let's look at what we just collected. This is the raw data — exactly as it appeared on the website.

In [ ]:
# Organise data into a table using pandas
df = pd.DataFrame(books_data)

print("First 10 products collected:")
print("=" * 60)
df.head(10)

In [ ]:
df.to_csv('books.csv', index = False)

---
## Step 3: Gemini API — Turning Data into Business Insights

Now the interesting part. We've **collected** the data. Now we'll send it to **Google's Gemini AI** and ask it real business questions.

This is what the analytics stage of the pipeline looks like in practice.

> 🔑 **You need a Gemini API key** — get one free at [aistudio.google.com](https://aistudio.google.com)

In [ ]:
# --- GEMINI API SETUP ---
# Replace the string below with your actual Gemini API key

GEMINI_API_KEY = ""  # 👈 Replace this!

# Configure the Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini API configured!")

In [ ]:
# Convert the full dataframe to a string to send to Gemini
data_string = df.to_string(index=False)

# Helper function to ask Gemini a question
def ask_gemini(question):
    prompt = f"""
    You are a business analyst for an eCommerce bookshop.

    Here is our full product catalogue data:
    {data_string}

    Please answer the following business question in plain English,
    with clear and practical recommendations:

    {question}

    Format your response using markdown with short paragraphs and bullet points where appropriate.
    Keep your answer concise and actionable.
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    display(Markdown(response.text))

Let's start with simple questions!

In [ ]:
q1 = "which book is the most expensive?"
print("Asking Gemini...\n")
ask_gemini(q1)

In [ ]:
q2 = "which book is the cheapest but with high rating?"
print("Asking Gemini...\n")
ask_gemini(q2)

### Business Question 1: Pricing Strategy
**"Are our prices competitive? Should we adjust our pricing strategy?"**

In [ ]:
question_1 = "Which price range has the most 5-star products — what does this tell us about our pricing strategy?"

print("Asking Gemini...\n")
ask_gemini(question_1)

### 📦 Business Question 2: Stock Recommendations
**"Which products should we promote or stock more of?"**

In [ ]:
question_2 = "Looking at customer ratings and prices, which types of products should we stock more of or promote on our homepage to maximise customer satisfaction and sales?"

print("Asking Gemini...\n")
ask_gemini(question_2)


### 🎯 Business Question 3: Your Own Question!
**Ask Gemini anything about the data — be creative!**

In [ ]:
# 👇 Change this question to anything you want to ask!
your_question = "Replace with your own questions"

print("🤖 Asking Gemini...\n")
your_answer = ask_gemini(your_question)

---
##  What just happened? — A simple AI-powered data pipeline

```
 🌐 COLLECT             💾 STORE              🤖 ANALYSE
Web Scraping   →   DataFrame (pandas)    Gemini AI insights
```

### Key takeaways:

1. **Data collection at scale is automated** — we collected 1000 products in seconds. At real eCommerce scale, this runs continuously, 24/7.

2. **APIs are the same idea, but structured** — instead of scraping a webpage, you ask a service for data in a clean, agreed format. Faster and more reliable.

3. **AI is just the last step in the pipeline** — Gemini is powerful, but it's only as good as the data you feed it. Garbage in, garbage out. Please note that here we just quickly demonstrate you the beauty of API. We are not encouraging you to use AI to do all the analyses for you. One potential risk is that you will be restricted to share data with AI directly under business context, and another limitation is that AI doesn't know the complex business context and they might not necessarily help you make good decisions!.